![hslu_logo.png](./img/hslu_logo.png)


<hr style="border:1px solid black">

<h1 style="text-align:center;font-size:50px"><b>AAI - FS25</b></h1>
<p style="text-align:center;font-size:40px">Week 06</p>

---
# CNN Architecture Optimization
---
---
# Table of contents for week 06
1. [Motivation](#motiv)
2. [CNN Complexity Analysis](#complex_analysis)
     1. [Complexity Metrics](#complex_metrics)
     2. [Complexity and Model Parameters](#complex_model)
     3. [Complexity and System Parameters](#complex_system)
     4. [Exercise 1: CNN Complexity Analysis](#complex_ex1)
3. [CNN Architecture Optimization](#optim_cnn_arch)
     1. [Evolutionary Architecture Search](#optim_eas)
         1. [Introduction to EAS](#optim_eas_intro)
         2. [User Guide for EAS](#optim_eas_ug)
     2. [Exercise 2: MNIST Optimization](#optim_ex2)
     3. [Exercise 3: CIFAR Optimization Challenge (**Semesterleistung 10 %**)](#optim_ex3)
4. [References](#reference)

### **Installation for SW06**

Perform the following before proceeding:

1. Unzip the file *eas.zip* directly in the directory SW06, such that the directory structure looks as follows:
   
<img src="./img/sw06_dir_structure.png" alt="Directory SW06" width="150px"/>

2. In the activated python environment, install some additional packages by running:

        pip install dask pygal cairosvg pyqt5 tf2onnx 

# Motivation <a name="motiv"></a>

Usually, when we speak about the **performance** of an AI model, we mean its ability to perform the task it was trained for. In case of an classifier, important metrics for this performance are accuracy, precision and recall.

However, there is another aspect of *performance* related to the question "How much computational effort is required to achieve this accuracy?". The higher the effort, the higher the requirements on processing performance of the computing platform used. The question regarding effort can be broken down into two parts:

- Computational effort required for model training, and
- Computational effort required for model inference.

Although the two types of effort are closely related since both scale with the complexity of the model, in the context of embedded (or edge) computing, the computational effort for model inference is often considered more important than the effort for training. This is because usually training is done only a limited number of times on a server environment, while inference is then continuously performed on the edge device throughout the run-time of the system. Therefore, in this course we will focus on the computational effort for model inference only.

After the historical breakthrough of Deep Learning with the success of AlexNet in 2012 in the ImageNet classification challenge, the question about computational effort and required processing performance has often been neglected or considered insignificant. As a result, the increase in model accuracy very often was bought by increased model complexity, see <a href="#fig1b">Fig. 1b</a>.

<center>
<img src="./img/ImgNet_Challenge.png" alt="Drawing" width="650px" />
</center>
<center>
<a id="fig1a">Fig. 1a:</a> Winning Deep Learning Architectures of ImageNet classification challenge with complexity metrics.
</center>
</br> 


This in turn lead to a situation ilustrated in <a href="#fig1b">Fig. 1b</a> below, when trying to apply Deep Learning methods to solve practical problems that must do without high-performance computing data centers.

<center>
<img src="./img/energy_1.png" alt="Drawing" width="500px" /> <sub>adapted from [1]</sub>
</center>
<center>
<a id="fig1b">Fig. 1b:</a> Early Prototype of a self-driving car dissipating 2.5 kW to process 200 MByte of data per second.
</center>
</br> 

However, in the last couple of years, a new paradigm and research area has been established at the intersection of Machine Learning and Edge Computing, often referred to as *Edge AI*, or *Green AI* [[2](#ref2)], or *TinyML* in case of deeply embedded systems [[3](#ref3)]. 

Following this trend, we will here focus on model complexity and associated computational effort for inference using CNNs.

# CNN Complexity Analysis <a name="complex_analysis"></a>

## Complexity Metrics <a name="complex_metrics"></a>

To measure the (inference) complexity of CNNs the following two metrics are usually considered:

| Metric    | Description |
|-----------|-------------|
| $N_W$     | \# of trainable parameters (Weights) in the model  |
| $N_{M}$   | \# of multiply-accumulate (MAC) operations per model inference |

It should be noted, that neither of these two metrics alone completely describes the complexity of a CNN. While $N_W$ better expresses the storage requirements of a CNN, $N_{MAC}$ gives the better indication on computational effort. This is due to the fact that dense (fully connected = FC) layers, in contrast to convolutional (CONV) layers, have fewer computations but more trainable parameters. As we will see below, this relative *parameter efficiency* of CONV layers should **not** be mistaken as a general advantage of CONV over FC layers with respect to model complexity.

On the other hand, $N_W$ and $N_{MAC}$ together provide a pretty comprehensive measure for the complexity of a CNN. Although there are other layers than CONV and FC in a typical CNN, these other layers are either not executed during inference on Edge devices (e.g. Batch Normalization layers) or their computational effort is negligible compared to the $N_{MAC}$ MAC-operations (e.g. Max Pooling layers typically following all except the last CONV layer). 

With respect to complexity, care should be taken with Activation layers. While some activation functions like ReLU are uncritical with respect to runtime requirements, other non-linear activation functions like Sigmoid can infer an enormous runtime overhead when executed with unoptimized software libraries. 

## Complexity and Model Parameters <a name="complex_model"></a>

### 2D Convolutional Layers

2D-convolution is a sliding window process. In each step over the input feature map of size $H\times W$, the filter coefficients (kernel weights) are multiplied element-wise with the underlying activations. All the partial products are accumulated to form one activation in the output feature map of size $E\times F$. The step size is called stride and determines the output feature map size. Very often for the filter size holds $R = S$, with typical values being (1), 3, 5, 7, 9, 11.

<img src="./img/dl_conv_layer_1.png" alt="Conv1" width="450px" />

If there is more than one input feature map, all feature maps are treated at once with a corresponding 3D filter kernel. (Note that the filter kernel is 3D, but the convolution operation is 2D!) Therefore, in every CONV layer, **the number of input feature maps always equals the number of filter channels C**. Typically $1 \le C \le 128$, with larger values occuring at later layers. $C = 1$ usually only occurs at the first CONV Layer in case of 2D-input (gray-scale images).

<img src="./img/dl_conv_layer_2.png" alt="Conv2" width="450px" />

Very often, more than one 3D-filter is applied by a CONV Layer. If $M$ filters are applied in CONV Layer $k$, then the resulting $M$ output feature maps of layer $k$ will turn into $C$ input feature maps of layer $k + 1$, i.e. $$M_k = C_{k+1}$$.

<img src="./img/dl_conv_layer_3.png" alt="Conv3" width="450px" />

Thus, the total number of MAC operations required by a CONV layer for one inference, can be computed by counting how many output activations are produced with which kernel volume, i.e. $$N^{\rm{CONV}}_{M}\ =\ (E\cdot F\cdot M)\cdot(R\cdot S\cdot C)$$ 

Note that this formular holds true independently of the stride and padding mode used. Also note that the formula neglects the $(E\cdot F\cdot M)$ ooperations required to add the $M$ different bias values to the convolution results.

The number of trainable parameters in a CONV layer is thus $$N^{\rm{CONV}}_{W}\ =\ (R\cdot S\cdot C)\cdot M\ +\ M $$ 

### Fully Connected Layers

The corresponding formulas for the number of MAC operations and number of weights of FC layers are $$N^{\rm{FC}}_{M}\ =\ L_{i}\cdot L_{o} $$ and $$N^{\rm{FC}}_{W}\ =\ L_{i}\cdot L_{o}\ +\ L_{o} $$ where $L_i$ and $L_o$ are the number of activations in the input and output feature map, respectively.

## Complexity and System Parameters <a name="complex_system"></a>

When using AI models on edge devices a natural question that arises is "How does model complexity influence system parameters?".

Obviously, computational effort is directly related to energy consumption. Therefore, for training and inference on server environments, computational effort is typically related to CO2 emission and heat dissipation. For model inference on embedded systems, computational effort is associated with the limited compute, memory, and power ressources of such systems. 

For instance, when performing CNN detection and classification on a video stream on an embedded camera, the complexity of the CNN used will influence the maximum frame rate that can be achieved. Or, when a CNN performs acoustic scene classification in a hearing aid, the efficiency of the CNN at work will influence battery lifetime. 

---

## Exercise 1: CNN Complexity Analysis <a name="complex_ex1"></a>

**[sw06.01_macs_weights.ipynb](./sw06.01_macs_weights.ipynb)**

This exercise uses the reference networks for the MNIST and CIFAR-10 usecases from <a href="#optim_ex_arch">below</a> as examples.

Hints:
- When calculating the number of MAC operations, start with the dimensions of the output feature map.  
- Compare your results with the numbers given in the table <a href="#optim_ex_arch">below</a>. 

---

# CNN Architecture Optimization <a name="optim_cnn_arch"></a>

The potential of any system optimization is typically higher the earlier in the system design cycle it is performed. The same is true for systems that perform AI model inference, in particular on embedded devices. Once the AI model type has been selected for a given task, the next step is to design the model architecture. This often is a manual, time-consuming process with no guarantee that the selected model architecture is efficient, not to mention optimal. Due to the complexity of the model and the high number of hyper-parameters to be selected, an exhaustive grid-search over all hype-parameter values is prohibitive.

One way to speed-up this search is to steer the hyper-parameter selection by means of a cost function. A particular way to minimize the cost function is to use an evolutionary algorithm, which builds on concepts known from evolution in nature. Here we will explore such evolutionary algorithm for optimizing CNN architectures, which was developed in a Master Thesis at HSLU and published in [[4](#ref4)].

## Evolutionary Architecture Search <a name="optim_eas"></a>

The basic idea of the evolutionary architecture search (EAS) algorithm is to find *better* CNNs starting from an initial, *reasonably good* network. Here, *better* could mean, for instance, a CNN with less MAC operations but no accuracy loss, or higher accuracy with as few additional MACs as possible.

### Introduction to EAS <a name="optim_eas_intro"></a>

#### Overview

The following <a href="#fig2">Fig. 2</a> depicts an overview of the EAS environment. It shows the main components of the EAS algorithm, which consists of the two main parts *Initialization* and *Evolution*, as well as the user interface based on python scripts. The most runtime-intensive parts of EAS are the training runs for the CNN architectures in the initial population and for the new architectures created from each population during evolution.

<center>
<img src="img/EAS_algorithm_aai.png" alt="EAS Algorithm" width="750" /></br> 
<a id="fig2">Fig 2:</a> EAS Environment
</center>

#### Terminology

**DNA**: Each network architecture that is created and trained during an EAS run, is associated with a unique DNA. The DNA contains information on the history (parents), performance, and complexity of the network (Keras model). The reference (parent) network is denoted DNA 0. A network can be selected as parent for a new network, which will be altered by a randomly selected mutation, e.g. an additional layer or different hyperparameter of an existing layer.

**Ranking**: Sorting of the current population of networks according to the defined optimization criteria. The higher the rank, the “better” the network. The following optimization criteria are supported in this version by setting configuration parameter *optim_metric* to:

- "acc" - networks sorted by maximum evaluation-accuracy achieved
- "macs" - networks sorted by minimum # of MACs
- "macs_and_acc" - networks sorted by a 2-dimensional ranking function, see below.

**Fitness Assignment**: The rank and the selective pressure SP [1..2] are used to calculate the fitness of the network. The higher the fitness, the higher the chance that the network is selected as parent.

**Selection**: Parents are selected with a method called stochastic universal sampling.

**Mutation**: On each selected parent networks, a certain number of mutations are performed to get networks with different DNA in the population. If a mutation leads to an invalid architectures or an architecture outside the search space, the created architecture is disposed immediately. 

**Training**: Newly created individuals are trained (FP32). The best achieved validation accuracy during training is stored in the DNA of each network.

**Reinsertion**: After the mutation step, the population has grown and must be limited to the defined population size again. At this point, all individuals are checked again for search space violation. The population is then sorted according the evaluation accuracy achieved by each DNA. The networks with the highes accuracy form the new population.

#### 2D Ranking Function

An important feature of this EAS enviornment is, that optimization for accuracy and # of MACs can be performed jointly.

Using this feature requires a two-dimensional ranking function, which allows to control the two optimization variables independently. For this purpose, a 2-dimensional elliptical Gaussian function
$$ f(x,y)\ =\ e^{-\left(\frac{(x-\mu_x)^2}{2\sigma_x^2} + \frac{(y-\mu_y)^2}{2\sigma_y^2}\right)}$$ 
with variates $X$ = # of MACs and $Y$ = accuracy [\%] has been choosen. By setting expected value $\mu$ and standard deviation $\sigma$ of the two variates, the ranking function can be easily adapted to the requirements of a specific usecase. 

<a href="#fig2">Fig. 3</a> below shows the 2D ranking function of usecase MNIST with the default parameters $\mu_x$ = 300'000 MACs, $\mu_y$ = 99 \%. 

<center>
<img src="img/mnist_ranking_default.png" alt="2D ranking fucntion" width="850" /></br> 
<a id="fig3">Fig 3:</a> 2D Gaussian ranking function
</center>

The following points should be taken into account when defining the 2D ranking function for a specific usecase:

1. The expected (center) value $\mu_x$ should be set lower than the # of MACs of the reference DNA 0, but not unrealistically low.
2. Similarly, the expected (center) value $\mu_y$ should be set higher than the accuracy of the reference DNA 0, but not unrealistically high.
3. The standard deviation $\sigma_x$ should be set, such that the gradient in x-direction reduces continuously towards zero at the search space boundary (max. # of MACs).
4. The standard deviation $\sigma_y$ should be set, such that the gradient in y-direction reduces continuously towards zero at the initial accuracy limit, and remains reasonably high as the accuracy limit is increased over the number of evolution rounds planned.

To support the design of a suitable 2D ranking function for a particular usecase, the following plotting function can be used:

      %run -m plot_rank_func MAC_0 ACC_0 MAC_T ACC_T

This function reads the search space constraints and the 2D Gaussian function parameters from the configuration file. In the generated contour plot the reference DNA 0 at position (MAC_0, ACC_0) and the ideal target DNA at position (MAC_T, ACC_T) are indicated for reference. 

### User Guide for EAS <a name="optim_eas_ug"></a>

#### Example Architectures <a name="optim_ex_arch"></a>

The following two classification usecases *Small* and *Large* have been prepared for experimentation with the EAS environment. <a href="#fig4">Fig. 4</a> shows the reference architectures of these two usecases.

|                                     | Small: MNIST         | Large: CIFAR-10      | 
| ------------------------------------|----------------------|----------------------|
| Reference Architecture (DNA 0)      | [Keras Example](https://keras.io/examples/vision/mnist_convnet/) | [Kaggle Example](https://www.kaggle.com/code/faressayah/cifar-10-images-classification-using-cnns-88)     |
| Total # of MAC operations         | 2'440'960            | 38'896'896           |
| Total # of train. parameters (weights)  | 34'826               | 552'362              |
| Training Dataset                    | 60'000 x 28 x 28 x 1 | 50'000 x 32 x 32 x 3 |
| Test Dataset                        | 10'000 x 28 x 28 x 1 | 10'000 x 32 x 32 x 3 |
| EAS runtime with default parameters | 2...5 min            | 11...12 h            |

<center>
<img src="img/MNIST_CIFAR_ref_architectures.png" alt="Reference Architectures" width="950" /> </br>
<a id="fig4">Fig 4:</a> CNN reference architecture (DNA 0) of MNIST (top) and CIFAR (bottom) usecase.
</center>
</br>

#### Configuration <a name="optim_eas_ug_config"></a>

The parameters for EAS can be adjusted in the [config file](config.py). The table below lists the most important paramters and their default values for the two usecases *Small* and *Large*, which can also be selected in the [config file](config.py):

| Parameter                  | Description                                   | MNIST  | CIFAR  |
| -------------------------- | ----------------------------------------------|--------|--------|
| pop_size                   | # of networks (DNAs) in one population        | 4      | 10     |
| nbr_of_evo_rounds          | # of evaluation rounds = # of populations -1  | 3      | 4      |
| nbr_of_selection_pick      | # of parents used to form new population      | 1      | 3      |
| nbr_of_children_per_parent | # of mutations per parent for new population  | 2      | 2      |
| nbr_of_training_epochs     | # of training epochs                          | 4      | 12     |
| batch_size                 | Training batch size paramter                  | 100    | 200    |
| train_dataset_size         | Size of the training dataset used in EAS      | 10000  | 30000  |
| val_dataset_size           | Size of the validation dataset                | 1000   | 3000   |
| use_early_stopping         | Early stopping, if training does not advance  | True   | True   |
| use_ReducedLR              | Dynamic learning rate over training epochs    | True   | True   |
| init_LR                    | Initial learning rate in 1st training epoche  | 0.0004 | 0.005  |

#### Running EAS

To start the EAS, run the following:

     %run -m main_evo

The process can take from several minutes to several hours or even days, depending on the network complexity and EAS configuration parameters. It is therefore good practice to perform initial experiments with the CNN reference architecture (DNA 0) to find a reasonable EAS configuration before starting the actual EAS optimization run, see [sw06.02_eas_MNIST.ipynb](./sw06.02_eas_MNIST.ipynb) for an example.

#### Runtime Estimation

The runtime (rt) of the full EAS is estimated during training of the initial population based on the formula:

$$ {rt\_EAS} = (pop\_size + (nbr\_of\_evo\_rounds * nbr\_of\_parents * nbr\_of\_children\_per\_parent)) * rt\_Training $$

- After training of each indidividual network, the runtime is estimated based on this training. The estimated runtime is printed to the console.
- After the complete initial population has been trained, the runtime is estimated based on the average individual training times. The estimated runtime is printed to the console and saved in the main_log.txt file.

At any point, e.g. if the estimated runtime is found too high, the EAS process can be stopped with <kbd>Ctrl</kbd> + <kbd>C</kbd>.

#### Result Logging

At the end of any run, the output of the EAS can be found in folder [log](log/). The most important log files are: 

| File Name (.txt) | Content
|------------------|-------------------------------------------------------------------------------------|
| main_log         | Overview of all EAS runs performed so far                                           |
| arch_log         | Detailed layer-architecture of all networks generated during the last EAS run       |
| dna_log          | DNA information (e.g. accuracy, MACs) of all networks generated during last EAS run |
| ranked_log       | Ranking (higher is better) of networks per population during last EAS run           |
| population_log   | DNA IDs selected for each population during last EAS run                            |

If a new EAS run is started, the log files together with the corresponding [config file](config.py) are saved in a new sub-fulder *OldLogsDATE* within folder [log](log/).

#### Analysis of EAS results

To obtain an overview of the best-performing networks found during the last EAS run, use the command 

      %run -m run_analysis NUM_DNA MIN_ACC

This will provide the DNA information (accuracy, # MAC, # of parameters) for the DNA_NUM best networks found during the last EAS run. Information is printed in the console and written to file "analysis.txt" in folder [log](log/) for three categories

- NUM_DNA networks with the highest accuracy
- NUM_DNA networks with the least # of MACs
- NUM_DNA networks with MIN_ACC accuracy and the least # of MACs

### Generate evolution plots

To visualize the evolution of networks during the last EAS run, execute the command

      %run -m plot_acc_macs NUM_GEN MIN_ACC

The script generates an interactive plot "CNN_evolution.svg" in folder [log](log/) showing all created networks that have an evaluation accuracy of at least MIN_ACC. The plot shows the seed DNA 0 (in red) and the initial population DNAs (in blue). For all subsequent generations the DNAs from NUM_GEN generations are grouped into the same color class.

<a href="#fig5">Fig. 5</a> shows two example evolutions for Usecase MNIST.

<center>
<img src="img/MNIST_evolution_short.jpeg" alt="MNIST_evolution_short" width="850" /> </br>
</center>
<center>
<img src="img/MNIST_evolution_long.jpeg" alt="MNIST_evolution_long" width="850" /></br>
</center>
<center>
<a id="fig5">Fig 5:</a> Evolution over 3 (top) and 30 (bottom) evolution rounds for usecase MNIST.
</center>
</br>

### Visualize CNN Architecture

The architecture of a selected model from the last EA run can be visualized by running

      %run -m visualize_cnn_arch DNA_NUM

A file "Architecture_DNA_3.svg" will be created in folder [log](log/).

### Evaluate and Retrain a CNN

Since the best trained model of each network is stored during the EAS, it's performance on the testset can be ebaluated after EAS has completed. To test the best trained model of CNN with ID (DNA) 3, run

      %run -m evaluate_model best_model_dna_3

After the testset accuracy is printed out, the user is asked if the model shall be retrained. The parameters for retraining can be adjusted in the [config file](config.py). If retraining is performed, the retrained model is then again evaluated on the testset and saved as "best_model_dna_3_retrnd" in folder [log/training_models](log/training_models).

### Export Model to ONNX

Any of the (re-)trained Keras models from the last EAS can be converted to onnx format, for instance

      %run -m python export_model best_model_dna_3_retrnd

The converted model will be stored under the same name in folder [log/training_models](log/training_models).


---

## Exercise 2: MNIST Optimization <a name="optim_ex2"></a>

**[sw06.02_eas_MNIST.ipynb](./sw06.02_eas_MNIST.ipynb)**

---

## Exercise 3: CIFAR Optimization (**Semesterleistung**)<a name="optim_ex3"></a>

**Termin:** 3. April 2025 12:20 auf ILIAS "Semesterleistung 1"

**Bewertung:** Max. 10 % der MEP Punkte

**Abgabe:** 1 bis 2 Seiten Bericht (pdf) zu Challenge 1 <ins>oder</ins> 2 mit
1. Interpretierter Plot der angepassten 2D Ranking Funktion
1. Interpretierter Evolution Plot mit mindestens 30 DNA's
1. Visualisierung der besten DNA Architektur
1. Berechnung per Layer der Anzahl MAC Operationen und Anzahl Weights der besten DNA
1. Interpretation der besten DNA Architektur im Vergleich zu DNA 0

**Challenge 1:** Wer findet das CIFAR Netzwerk mit
- Testset accuracy > 78 %
- und min. Anzahl von MAC Operationen?

**Challenge 2:** Wer findet das CIFAR Netzwerk mit
- Anzahl MACs < 50'000'000
- und max. Testset Accuracy?

---

# References <a name="reference"></a>
1. <a name="ref1"></a> Self-Driving Cars Use Crazy Amounts of Power. [Online Article](https://www.wired.com/story/self-driving-cars-power-consumption-nvidia-chip/).
2. <a name="ref2"></a> AI in the 2020s Must Get Greener. [IEEE Spectrum](https://spectrum.ieee.org/energy-efficient-green-ai-strategies).
3. <a name="ref3"></a> The Rise of TinyML: Revolutionizing Edge AI [Online Article](https://qksgroup.com/blogs/the-rise-of-tinyml-revolutionizing-edge-ai-with-compact-machine-learning-590).
4. <a name="ref4"></a> Efficient Evolutionary Architecture Search for CNN Optimization on GTSRB. [IEEE ICMLA 2019](https://ieeexplore.ieee.org/document/8999305)